# Compile Results Across All Schemes

Consolidates full-cohort and feature-comparison results across:
- **death_met**: Death + 7 metastasis sites
- **icd3_post**: ICD-10 level 3 codes (post-treatment)
- **icd4_post**: ICD-10 level 4 codes (post-treatment)
- **phecode_post**: Phecodes (post-treatment)

Outputs a single DataFrame with c-index, mean AUC(t), and per-feature metrics for every event across all schemes.

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

CWD = Path.cwd().resolve()
if (CWD / 'jupyter_notebooks' / 'metrics').exists():
    REPO_ROOT = CWD
elif CWD.name == 'metrics' and (CWD / 'compile_all_scheme_results.ipynb').exists():
    REPO_ROOT = CWD.parents[1]
else:
    REPO_ROOT = CWD

METRICS_DIR = REPO_ROOT / 'jupyter_notebooks' / 'metrics'
MODEL_TRAINING_DIR = REPO_ROOT / 'python_scripts' / 'model_training'
for path in (METRICS_DIR, MODEL_TRAINING_DIR):
    if path.exists() and str(path) not in sys.path:
        sys.path.insert(0, str(path))

from publication_plot_utils import (
    plot_base_vs_text_scatter,
    plot_delta_c_index_by_scheme,
    plot_feature_comparison_distribution,
    plot_model_performance_by_category,
    plot_publication_summary,
    plot_top_text_gain_events,
    set_publication_style,
)

try:
    import icd10
    HAS_ICD10 = True
except ImportError:
    HAS_ICD10 = False
    print('icd10 package not available; ICD descriptions will be codes only')

In [ ]:
# === Paths ===
DATA_PATH = '/data/gusev/USERS/jpconnor/data/clinical_text_embedding_project/'
CODE_PATH = os.path.join(DATA_PATH, 'code_data/')
SURV_PATH = os.path.join(DATA_PATH, 'time-to-event_analysis/')
RESULTS_BASE = os.path.join(SURV_PATH, 'results/')
INTAE_DATA_PATH = '/data/gusev/PROFILE/CLINICAL/robust_VTE_pred_project_2025_03_cohort/data/'

# Output
COMPILED_PATH = os.path.join(RESULTS_BASE, 'compiled_all_schemes/')
PUBLICATION_PATH = os.path.join(COMPILED_PATH, 'publication_figures/')
os.makedirs(COMPILED_PATH, exist_ok=True)
os.makedirs(PUBLICATION_PATH, exist_ok=True)

# Scheme config (mirrors slurm_array_utils.SCHEME_CONFIG)
SCHEMES = {
    'death_met': 'death_met_results',
    'icd3_post': 'level_3_ICD_post_results',
    'icd4_post': 'level_4_ICD_post_results',
    'phecode_post': 'phecode_post_results',
}

FEATURE_NAMES = ['stage', 'treatment', 'labs', 'somatic', 'prs', 'text']
PREVALENCE_THRESHOLDS = [0.01, 0.025, 0.05]
EXCLUDED_EVENTS = {'vte'}

MET_EVENTS = {'brainM', 'boneM', 'adrenalM', 'liverM', 'lungM', 'nodeM', 'peritonealM'}
MET_DESCRIPTIONS = {
    'brainM': 'Brain metastasis', 'boneM': 'Bone metastasis',
    'adrenalM': 'Adrenal metastasis', 'liverM': 'Liver metastasis',
    'lungM': 'Lung metastasis', 'nodeM': 'Lymph node metastasis',
    'peritonealM': 'Peritoneal metastasis',
}

In [ ]:
def get_event_description(event, scheme):
    """Look up a human-readable description for an event code."""
    if event == 'death':
        return 'Death (overall survival)'
    if event == 'vte':
        return 'Venous thromboembolism'
    if event in MET_DESCRIPTIONS:
        return MET_DESCRIPTIONS[event]
    if scheme in ('icd3_post', 'icd4_post') and HAS_ICD10:
        # Try exact match first, then level-3 prefix
        if icd10.exists(event):
            return icd10.find(event).description
        prefix = event.split('.')[0] if '.' in event else event
        if icd10.exists(prefix):
            return f'{icd10.find(prefix).description} ({event})'
    # For phecodes or unresolved ICD codes, return the code itself
    return event


def get_event_category(event, scheme):
    """Classify event into a broad category for plotting."""
    if event == 'death':
        return 'Death'
    if event in MET_EVENTS:
        return 'Metastasis'
    if event == 'vte':
        return 'VTE'
    category_map = {
        'icd3_post': 'icd3_post',
        'icd4_post': 'icd4_post',
        'phecode_post': 'phecode_post',
    }
    return category_map.get(scheme, scheme)


def get_icd_chapter(event, scheme):
    """Get ICD chapter info if applicable."""
    if scheme in ('icd3_post', 'icd4_post') and HAS_ICD10:
        code_to_check = event.split('.')[0] if '.' in event else event
        if icd10.exists(code_to_check):
            try:
                code = icd10.find(code_to_check)
                return code.chapter, code.block_description
            except Exception:
                return None, None
    return None, None


def _dedupe_in_order(values):
    seen = set()
    out = []
    for value in values:
        if value not in seen:
            out.append(value)
            seen.add(value)
    return out


def _normalize_icd10_undotted(code):
    if pd.isna(code):
        return None
    code = str(code).strip().upper()
    code = ''.join(ch for ch in code if ch.isalnum() or ch == '.')
    code = code.replace('.', '')
    return code if code else None


def _to_icd10_level_3(code):
    code = _normalize_icd10_undotted(code)
    if code is None or len(code) < 3:
        return None
    return code[:3]


def _to_icd10_level_4(code):
    code = _normalize_icd10_undotted(code)
    if code is None or len(code) < 3:
        return None
    if len(code) == 3:
        return code
    return f'{code[:3]}.{code[3]}'


def _normalize_phecode(code):
    if pd.isna(code):
        return None
    code = ''.join(ch for ch in str(code).strip() if ch.isdigit() or ch == '.')
    if not code:
        return None
    if code.count('.') > 1:
        left, right = code.split('.', 1)
        code = f"{left}.{right.replace('.', '')}"
    if '.' in code:
        left, right = code.split('.', 1)
        right = right.rstrip('0')
        return left if right == '' else f'{left}.{right}'
    return code


def _resolve_column(df, expected):
    col_map = {col.strip().lower(): col for col in df.columns}
    if expected not in col_map:
        raise ValueError(f"Expected column '{expected}' not found. Available columns: {list(df.columns)}")
    return col_map[expected]


_ICD10_EXCLUDED_PREFIXES = {'V', 'W', 'X', 'Y', 'O', 'C'}


def _is_excluded_icd10(code):
    if not code:
        return True
    first = code[0].upper()
    if first in _ICD10_EXCLUDED_PREFIXES:
        return True
    if first == 'D' and len(code) >= 3:
        try:
            num = int(code[1:3])
            if num <= 49:
                return True
        except ValueError:
            pass
    return False


_PHECODE_EXCLUDED_RANGES = [(140, 239.99), (635, 677.99), (800, 999.99)]


def _is_excluded_phecode(code):
    if not code:
        return True
    try:
        val = float(code)
    except (ValueError, TypeError):
        return True
    return any(lo <= val <= hi for lo, hi in _PHECODE_EXCLUDED_RANGES)


def _threshold_to_label(threshold):
    return f"{threshold * 100:g}".replace('.', '_') + 'pct'


def _compute_prevalence_stats(event_data, cohort_mrns, event_col, events_to_analyze, scheme):
    cohort_size = len(cohort_mrns)
    mask = event_data['DFCI_MRN'].isin(cohort_mrns)
    counts = (
        event_data.loc[mask]
        .drop_duplicates(subset=['DFCI_MRN', event_col])[event_col]
        .value_counts()
    )

    records = []
    for event in _dedupe_in_order([event for event in events_to_analyze if pd.notna(event)]):
        prevalence_n = int(counts.get(event, 0))
        prevalence_rate = prevalence_n / cohort_size if cohort_size else np.nan
        records.append({
            'scheme': scheme,
            'event': event,
            'prevalence_n': prevalence_n,
            'prevalence_rate': prevalence_rate,
            'prevalence_filter_applies': True,
        })
    return pd.DataFrame(records)

## Compile full-cohort + feature-comparison metrics across all schemes

In [ ]:
all_rows = []

for scheme, results_dir in SCHEMES.items():
    full_cohort_path = os.path.join(RESULTS_BASE, results_dir, 'full_cohort')
    feature_comps_path = os.path.join(RESULTS_BASE, results_dir, 'feature_comps')

    if not os.path.isdir(full_cohort_path):
        print(f'  Skipping {scheme}: {full_cohort_path} not found')
        continue

    # Find events that have full-cohort results
    events = [e for e in os.listdir(full_cohort_path)
              if os.path.isdir(os.path.join(full_cohort_path, e))]

    print(f'{scheme}: {len(events)} events found')

    for event in events:
        row = {'scheme': scheme, 'event': event}

        # --- Event metadata ---
        row['event_description'] = get_event_description(event, scheme)
        row['event_category'] = get_event_category(event, scheme)
        row['icd_chapter'], row['icd_block_description'] = get_icd_chapter(event, scheme)

        # --- Full cohort: base and text models ---
        event_path = os.path.join(full_cohort_path, event)
        for model_name, file_name in [('base', 'base_test.csv'), ('text_full_cohort', 'text_test.csv')]:
            fpath = os.path.join(event_path, file_name)
            if os.path.isfile(fpath):
                df = pd.read_csv(fpath)
                for metric in ['mean_c_index', 'mean_auc(t)']:
                    if metric in df.columns:
                        row[f'{model_name}_{metric}'] = df[metric].values[0]

        # --- Feature comparisons ---
        feat_event_path = os.path.join(feature_comps_path, event)
        if os.path.isdir(feat_event_path):
            for feature in FEATURE_NAMES:
                prefix = f'{feature}_feat_comp' if feature == 'text' else feature
                feat_file = os.path.join(feat_event_path, f'{feature}_test.csv')
                if os.path.isfile(feat_file):
                    df = pd.read_csv(feat_file)
                    for metric in ['mean_c_index', 'mean_auc(t)']:
                        if metric in df.columns:
                            row[f'{prefix}_{metric}'] = df[metric].values[0]

        all_rows.append(row)

results_df = pd.DataFrame(all_rows)
print(f'\nTotal: {len(results_df)} event-scheme rows')
results_df['scheme'].value_counts()

In [ ]:
# Remove VTE from compiled metrics / plots
n_excluded = int(results_df['event'].isin(EXCLUDED_EVENTS).sum())
if n_excluded:
    print(f'Excluding {n_excluded} rows for {sorted(EXCLUDED_EVENTS)} from compiled metrics and plots')
results_df = results_df.loc[~results_df['event'].isin(EXCLUDED_EVENTS)].copy()

# Compute improvement from adding text embeddings (full cohort)
results_df['delta_c_index'] = results_df['text_full_cohort_mean_c_index'] - results_df['base_mean_c_index']
results_df['delta_mean_auc'] = results_df['text_full_cohort_mean_auc(t)'] - results_df['base_mean_auc(t)']

# ------------------------------------------------------------------
# Annotate compiled results with raw-cohort prevalence, mirroring the
# original preprocessing logic used in generate_embedding_prediction_datasets.py
# ------------------------------------------------------------------
cohort_mrns = set(pd.read_csv(os.path.join(INTAE_DATA_PATH, 'follow_up_vte_df_cohort.csv'), usecols=['DFCI_MRN'])['DFCI_MRN'])
split_ehr_icd_subset = pd.read_csv(os.path.join(SURV_PATH, 'timestamped_icd_info.csv'))

icd_data_base = split_ehr_icd_subset.copy()
icd_data_base['ICD10_LEVEL_3_CD'] = icd_data_base['DIAGNOSIS_ICD10_CD'].map(_to_icd10_level_3)
icd_data_base['ICD10_LEVEL_4_CD'] = icd_data_base['DIAGNOSIS_ICD10_CD'].map(_to_icd10_level_4)
icd_data_base = icd_data_base.dropna(subset=['ICD10_LEVEL_3_CD']).copy()
icd_data_base = icd_data_base.loc[~icd_data_base['ICD10_LEVEL_3_CD'].map(_is_excluded_icd10)].copy()

icd3_prevalence_df = _compute_prevalence_stats(
    event_data=icd_data_base,
    cohort_mrns=cohort_mrns,
    event_col='ICD10_LEVEL_3_CD',
    events_to_analyze=results_df.loc[results_df['scheme'] == 'icd3_post', 'event'].tolist(),
    scheme='icd3_post',
)

icd4_data_base = icd_data_base.dropna(subset=['ICD10_LEVEL_4_CD']).copy()
icd4_prevalence_df = _compute_prevalence_stats(
    event_data=icd4_data_base,
    cohort_mrns=cohort_mrns,
    event_col='ICD10_LEVEL_4_CD',
    events_to_analyze=results_df.loc[results_df['scheme'] == 'icd4_post', 'event'].tolist(),
    scheme='icd4_post',
)

mapping_file = os.path.join(CODE_PATH, 'icd10_to_phecode_mapping.csv')
mapping_df = pd.read_csv(mapping_file)
mapping_icd_col = _resolve_column(mapping_df, 'icd10_code')
mapping_phecode_col = _resolve_column(mapping_df, 'phecode')
mapping_df['ICD10_NORM'] = mapping_df[mapping_icd_col].map(_normalize_icd10_undotted)
mapping_df['PHECODE'] = mapping_df[mapping_phecode_col].map(_normalize_phecode)
mapping_df = mapping_df.dropna(subset=['ICD10_NORM', 'PHECODE']).drop_duplicates(subset=['ICD10_NORM', 'PHECODE'])

phe_data = split_ehr_icd_subset.copy()
phe_data['ICD10_NORM'] = phe_data['DIAGNOSIS_ICD10_CD'].map(_normalize_icd10_undotted)
phe_data = phe_data.dropna(subset=['ICD10_NORM'])
phe_data = phe_data.merge(mapping_df[['ICD10_NORM', 'PHECODE']], on='ICD10_NORM', how='inner')
phe_data = phe_data.loc[~phe_data['PHECODE'].map(_is_excluded_phecode)].copy()

phecode_prevalence_df = _compute_prevalence_stats(
    event_data=phe_data,
    cohort_mrns=cohort_mrns,
    event_col='PHECODE',
    events_to_analyze=results_df.loc[results_df['scheme'] == 'phecode_post', 'event'].tolist(),
    scheme='phecode_post',
)

death_met_prevalence_df = pd.DataFrame({
    'scheme': 'death_met',
    'event': sorted(results_df.loc[results_df['scheme'] == 'death_met', 'event'].unique()),
    'prevalence_n': np.nan,
    'prevalence_rate': np.nan,
    'prevalence_filter_applies': False,
})

prevalence_annotation_df = pd.concat(
    [death_met_prevalence_df, icd3_prevalence_df, icd4_prevalence_df, phecode_prevalence_df],
    ignore_index=True,
)

results_df = results_df.merge(
    prevalence_annotation_df,
    on=['scheme', 'event'],
    how='left',
)
results_df['prevalence_filter_applies'] = results_df['prevalence_filter_applies'].fillna(False)

compiled_metrics_by_threshold = {}
compiled_event_sets_by_threshold = {}
prevalence_summary_rows = []
event_set_cols = [
    'scheme', 'event', 'event_description', 'event_category',
    'prevalence_filter_applies', 'prevalence_n', 'prevalence_rate',
]

cohort_size = len(cohort_mrns)

for threshold in PREVALENCE_THRESHOLDS:
    label = _threshold_to_label(threshold)
    pass_col = f'passes_prevalence_{label}'
    min_patients = max(1, int(threshold * cohort_size))

    results_df[pass_col] = (
        ~results_df['prevalence_filter_applies']
    ) | (results_df['prevalence_n'].fillna(0) >= min_patients)

    filtered_metrics = results_df.loc[results_df[pass_col]].copy()
    filtered_events = (
        filtered_metrics[event_set_cols]
        .assign(
            prevalence_filter=label,
            prevalence_threshold=threshold,
            min_patients_required=min_patients,
        )
        .sort_values(['scheme', 'event'])
        .reset_index(drop=True)
    )

    compiled_metrics_by_threshold[label] = filtered_metrics
    compiled_event_sets_by_threshold[label] = filtered_events

    filtered_metrics.to_csv(
        os.path.join(COMPILED_PATH, f'all_schemes_compiled_metrics_{label}.csv'),
        index=False,
    )
    filtered_events.to_csv(
        os.path.join(COMPILED_PATH, f'events_to_plot_{label}.csv'),
        index=False,
    )

    for scheme in SCHEMES:
        prevalence_summary_rows.append({
            'prevalence_filter': label,
            'threshold': threshold,
            'min_patients': min_patients,
            'scheme': scheme,
            'n_events': int(filtered_events.loc[filtered_events['scheme'] == scheme, 'event'].nunique()),
        })

prevalence_summary_df = pd.DataFrame(prevalence_summary_rows)
prevalence_annotation_df.to_csv(
    os.path.join(COMPILED_PATH, 'compiled_event_prevalence_annotations.csv'),
    index=False,
)
prevalence_summary_df.to_csv(
    os.path.join(COMPILED_PATH, 'prevalence_filter_event_counts.csv'),
    index=False,
)

PLOT_PREVALENCE_FILTER = '1pct'
plot_results_df = compiled_metrics_by_threshold[PLOT_PREVALENCE_FILTER].copy()
print(f'Using prevalence filter: {PLOT_PREVALENCE_FILTER} ({len(plot_results_df)} events)')

# Save the full compiled table with prevalence annotations / flags
results_df.to_csv(os.path.join(COMPILED_PATH, 'all_schemes_compiled_metrics.csv'), index=False)
print(f'Saved to {os.path.join(COMPILED_PATH, "all_schemes_compiled_metrics.csv")}')
print('Saved threshold-specific event lists:')
for threshold in PREVALENCE_THRESHOLDS:
    label = _threshold_to_label(threshold)
    print(f"  {label}: {os.path.join(COMPILED_PATH, f'events_to_plot_{label}.csv')}")

prevalence_summary_df

## Summary statistics by scheme

In [ ]:
summary = (plot_results_df
    .groupby('scheme')
    .agg(
        n_events=('event', 'count'),
        mean_base_c_index=('base_mean_c_index', 'mean'),
        mean_text_c_index=('text_full_cohort_mean_c_index', 'mean'),
        mean_delta_c_index=('delta_c_index', 'mean'),
        median_delta_c_index=('delta_c_index', 'median'),
        pct_improved=('delta_c_index', lambda x: (x > 0).mean() * 100),
    )
    .round(4)
)
summary

## Visualizations

In [ ]:
set_publication_style()

SCHEME_ORDER = list(SCHEMES)
CATEGORY_ORDER = ['Death', 'Metastasis', 'icd3_post', 'icd4_post', 'phecode_post']
PLOT_PUBLICATION_PATH = os.path.join(PUBLICATION_PATH, PLOT_PREVALENCE_FILTER)
os.makedirs(PLOT_PUBLICATION_PATH, exist_ok=True)

print(f'Saving publication figures to {PLOT_PUBLICATION_PATH}')
print(f'Plotting prevalence-filtered subset: {PLOT_PREVALENCE_FILTER}')


In [ ]:
plot_publication_summary(
    results_df=plot_results_df,
    output_dir=PLOT_PUBLICATION_PATH,
    scheme_order=SCHEME_ORDER,
    category_order=CATEGORY_ORDER,
    feature_names=FEATURE_NAMES,
)


In [ ]:
plot_delta_c_index_by_scheme(
    results_df=plot_results_df,
    output_dir=PLOT_PUBLICATION_PATH,
    scheme_order=SCHEME_ORDER,
)

plot_base_vs_text_scatter(
    results_df=plot_results_df,
    output_dir=PLOT_PUBLICATION_PATH,
    scheme_order=SCHEME_ORDER,
)

plot_model_performance_by_category(
    results_df=plot_results_df,
    output_dir=PLOT_PUBLICATION_PATH,
    category_order=CATEGORY_ORDER,
)

plot_feature_comparison_distribution(
    results_df=plot_results_df,
    output_dir=PLOT_PUBLICATION_PATH,
    feature_names=FEATURE_NAMES,
)


In [ ]:
plot_top_text_gain_events(
    results_df=plot_results_df,
    output_dir=PLOT_PUBLICATION_PATH,
    top_n=15,
)


## Top events by text improvement

In [ ]:
top_improved = (plot_results_df
    .dropna(subset=['delta_c_index'])
    .sort_values('delta_c_index', ascending=False)
    .head(25)
    [['scheme', 'event', 'event_description', 'event_category',
      'base_mean_c_index', 'text_full_cohort_mean_c_index', 'delta_c_index']]
    .round(3)
)
top_improved

In [ ]:
# Top 5 most improved per scheme
for scheme in SCHEMES:
    subset = plot_results_df.loc[plot_results_df['scheme'] == scheme].dropna(subset=['delta_c_index'])
    top5 = subset.nlargest(5, 'delta_c_index')[['event', 'event_description', 'base_mean_c_index', 'text_full_cohort_mean_c_index', 'delta_c_index']].round(3)
    print(f'\n--- {scheme} (top 5 improved) ---')
    print(top5.to_string(index=False))

In [ ]:
# Death + metastasis summary
death_met = plot_results_df.loc[plot_results_df['scheme'] == 'death_met'].copy()
if not death_met.empty:
    print('Death + Metastasis results:')
    print(death_met[['event', 'event_description', 'base_mean_c_index', 'text_full_cohort_mean_c_index', 'delta_c_index']]
          .sort_values('text_full_cohort_mean_c_index', ascending=False)
          .round(3)
          .to_string(index=False))

## Skipped Events Report

Events that were excluded at training time due to insufficient data (too few events or non-events for stratified 5-fold CV) vs events that failed due to runtime errors.

In [ ]:
import json
from glob import glob

SKIP_REPORT_DIR = os.path.join(RESULTS_BASE, 'skipped_events/')

# --- Load all skip reports ---
skip_files = sorted(glob(os.path.join(SKIP_REPORT_DIR, '*.jsonl')))
skip_rows = []
for fp in skip_files:
    with open(fp) as f:
        for line in f:
            line = line.strip()
            if line:
                skip_rows.append(json.loads(line))

if skip_rows:
    skipped_df = pd.DataFrame(skip_rows)
    print(f'Total skipped entries: {len(skipped_df)}')
    print(f'Unique skipped (scheme, event) pairs: {skipped_df.groupby(["scheme", "event"]).ngroups}')

    # Categorize skip reasons
    skipped_df['skip_category'] = skipped_df['reason'].apply(
        lambda r: 'too_few_events' if 'positive events' in r
        else 'too_few_non_events' if 'censored observations' in r
        else 'no_valid_rows' if 'No rows' in r
        else 'other'
    )

    print('\n--- Skip reasons by category ---')
    print(skipped_df['skip_category'].value_counts().to_string())

    print('\n--- Skipped events by scheme ---')
    print(skipped_df.groupby(['scheme', 'skip_category']).size()
          .unstack(fill_value=0).to_string())
else:
    skipped_df = pd.DataFrame()
    print('No skip reports found.')

In [ ]:
# --- Compare: expected vs completed vs skipped ---
# Load embedding prediction DFs to get the full list of events per scheme
from slurm_array_utils import SCHEME_CONFIG, load_embedding_prediction_df, get_events_from_df

coverage_rows = []
for scheme in SCHEMES:
    try:
        pred_df = load_embedding_prediction_df(scheme)
        all_events = {event for event in get_events_from_df(pred_df) if event not in EXCLUDED_EVENTS}
    except Exception as e:
        print(f'  Could not load events for {scheme}: {e}')
        continue

    completed_events = set(results_df.loc[results_df['scheme'] == scheme, 'event'])
    if not skipped_df.empty:
        skipped_events = set(skipped_df.loc[skipped_df['scheme'] == scheme, 'event']) - EXCLUDED_EVENTS
    else:
        skipped_events = set()

    # Events in the data but with no results AND no skip report = failed runs
    failed_events = all_events - completed_events - skipped_events

    coverage_rows.append({
        'scheme': scheme,
        'total_events': len(all_events),
        'completed': len(completed_events),
        'skipped_data': len(skipped_events),
        'failed': len(failed_events),
    })

    if failed_events:
        print(f'\n{scheme}: {len(failed_events)} FAILED events (no results, no skip report):')
        for e in sorted(failed_events)[:20]:
            print(f'  {e}')
        if len(failed_events) > 20:
            print(f'  ... and {len(failed_events) - 20} more')

coverage_df = pd.DataFrame(coverage_rows)
coverage_df['pct_completed'] = (100 * coverage_df['completed'] / coverage_df['total_events']).round(1)
print('\n--- Event coverage by scheme ---')
coverage_df